<a href="https://colab.research.google.com/github/recursive-factory/opencode-proxy/blob/main/notebooks/opencode-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OpenCode Free-Model Proxy — easy control panel 👋

This notebook runs our proxy on Google's computers and gives it a **public address**, so apps like Cursor, VS Code, or pi.dev can use OpenCode's free AI models.

**No coding, and no play button.** The panel cell below is set to **Run on startup**. When this session connects, Colab runs that cell by itself: it sets everything up, runs the tests, and starts your public address. The address appears in a green box; put it in your app with API key `not-needed`.

_(The same steps are still on the panel if you need to run them again: **🚀 One-click**, or 1 · Setup → 2 · Run tests → 3 · Start public address.)_

**Want the same address every time (recommended)?** One-time only: click the 🔑 key icon in Colab's left sidebar → **Secrets** → add `ZROK_TOKEN` (from your free zrok.io account) and `ZROK_NAME` (the name you want in your address, e.g. `my-proxy`), and turn on **Notebook access**. There is no built-in default name — your address will be `https://<ZROK_NAME>.shares.zrok.io`.

**Is the GitHub repo private?** Then also add `OC_PROXY_GITHUB_REPO_TOKEN` in Secrets — a read-only GitHub token for this repo — or the panel cannot download the code.

⚠️ Testing and chatting use a little of this session's **free allowance**. If things start failing with "quota" errors, restart the session (menu: **Runtime → Restart session**). The panel runs again on its own — that refreshes the allowance.


In [1]:
# Control panel. Colab runs this cell on its own when the session connects (Run on startup). No play button.
import urllib.request
try:
    from google.colab import userdata
    try:
        _tok = (userdata.get('OC_PROXY_GITHUB_REPO_TOKEN') or '').strip()
    except Exception:
        _tok = (userdata.get('GITHUB_TOKEN') or '').strip()
except Exception:
    _tok = ''
_req = urllib.request.Request(
    'https://api.github.com/repos/recursive-factory/opencode-proxy/contents/notebooks/colab_panel.py',
    headers={'Accept': 'application/vnd.github.raw'})
if _tok:
    _req.add_header('Authorization', 'Bearer ' + _tok)
exec(urllib.request.urlopen(_req).read().decode())


## Help & notes

- **Stop:** the red **Stop everything** button. Your public address stops working at once.
- **"Quota" / 429 errors:** this session's free allowance is used up. Menu: **Runtime → Restart session**. The panel cell runs again on startup. Your zrok address stays the same.
- **Your settings live in Secrets** (🔑 left sidebar): `OC_PROXY_GITHUB_REPO_TOKEN` (a read-only GitHub token — needed only if the repo is private), plus `ZROK_TOKEN` and `ZROK_NAME` (both needed for zrok; `ZROK_NAME` is your chosen address name — there is no default). Your address is `https://<ZROK_NAME>.shares.zrok.io`.
- **Safety:** anyone who has your public address can use it — there is no password. Don't post it publicly, and stop the tunnel when you're done.
- **Limit alert:** if the OpenCode free limit is hit, a red banner appears at the top of the panel (and the status in Step 3 shows the same alert). It fires once, then clears itself when a request works again. With the Tampermonkey auto-switch script installed, that banner also triggers an automatic session switch.
- **Lighter requests (2026-10-25):** the proxy no longer adds the 13k OpenCode prompt to every request — your app's own prompt plus a tiny gate stub is enough (verified in Colab; see the results cell below). Requests are a little cheaper and your app's instructions are followed more cleanly. Nothing changes for you: same buttons, same address.
- **OpenCode mode:** ON by default in every new session (Step 3 toggle). The request body is posted upstream as received (no system-prompt rewrite, no tool rename) — right for OpenCode itself. Turn it off for Cursor, VS Code, pi.dev, and the notebook Ask box; an explicit off is saved in this browser, anything else (new session, new browser, cleared storage) starts on again.
- **For developers:** the panel is `notebooks/colab_panel.py`; the proxy code is in `src/`. The old step-by-step cells were replaced by this panel on 2026-10-25.


## ✅ Minimal-system proxy — verified results (2026-10-25)

The upstream gate only checks that the system prompt contains an `<env>` block with a `Working directory:` line — not the OpenCode head, tool catalog, or skills (found via a custom OpenCode agent capture + ablation on a fresh Colab IP). The proxy therefore sends **your app's system prompt + a ~60-char gate stub** (`PROXY_SYSTEM=minimal`, the default; `PROXY_SYSTEM=full` restores the old 13k prepend, and a minimal-mode 429 auto-retries once with the full prompt).

End-to-end through the proxy with Cline-style requests, on a fresh Colab runtime:

| Test | Result | Prompt tokens |
|---|---|---|
| Plain chat | ✅ 200 | 408 (was ~2,400) |
| Tool call (`read_file` → `read` upstream → `read_file` back) | ✅ 200 | 422 |
| Streaming | ✅ 200 | — |
| No system message (57-char stub alone) | ✅ 200 | 311 |
| DeepSeek free model | ✅ 200 | 348 |
| Multi-turn tool-result round-trip | ✅ 200, correct answer | 416 |

**For developers re-testing:** one earlier session ran out of free quota mid-matrix (full-prompt control also 429 = quota, not shape). On a fresh runtime: run the panel (Setup pulls the latest `src/` from GitHub), then send one request from your app and check the proxy log shows `sys=minimal`.